# 4. Compiling to native trapped-ion gates

This is Appendix B. The question: how many *physical pulses* does the ansatz
cost on the universal trapped-ion qudit processor of Ringbauer *et al.*
(Nat. Phys. **18**, 1053, 2022)?

Their native set is

$$R^{(i,j)}(\theta,\varphi) = e^{-i\frac\theta2\sigma_\varphi^{(i,j)}},
\qquad
\mathrm{MS}^{(i,j)}(\theta,\varphi) = e^{-i\frac\theta4\left(\sigma_\varphi^{(i,j)}\otimes I + I\otimes\sigma_\varphi^{(i,j)}\right)^2},$$

a rotation inside one *pair of levels* of a single ion, and a pairwise
Mølmer–Sørensen gate.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np

import scipy.linalg as sla
from funciones.utilidades_ringbauer import descomponer
from funciones.utilidades_gellmann import GELLMANN
from funciones.utilidades import Jx1, Jz1

# Algoritmo 1 del apendice de Ringbauer et al., tal cual publicado: dos fases,
# y solo rotaciones entre niveles ADYACENTES.
print(f"{'generator':16s}{'phase 1':>9s}{'phase 2':>9s}{'total':>7s}")
casos = [("lambda_1", GELLMANN[1]), ("lambda_3", GELLMANN[3]),
         ("lambda_4", GELLMANN[4]), ("lambda_8", GELLMANN[8]),
         ("L_x", Jx1.full()), ("L_z", Jz1.full())]
for nom, M in casos:
    d = descomponer(sla.expm(-1j * 0.37 * np.asarray(M, complex)), adyacentes=True)
    print(f"  {nom:14s}{d['fase1']:>9d}{d['fase2']:>9d}{d['total']:>7d}")

generator         phase 1  phase 2  total
  lambda_1              1        0      1
  lambda_3              0        3      3
  lambda_4              3        0      3
  lambda_8              0        6      6
  L_x                   3        0      3
  L_z                   0        6      6


Two things are worth reading off that table.

**Diagonal generators are not free.** Phase 2 of the algorithm charges *three*
physical pulses per independent relative phase — there is no virtual-Z here. So
$\lambda_3$ costs 3 and $\lambda_8$ costs 6.

**$\lambda_4$ and $\lambda_5$ cost more than the other off-diagonal ones.** They
live on the level pair $(0,2)$, and the algorithm only uses adjacent pairs, so
they have to be routed through level 1.

## Why Gell-Mann is the natural basis here

The six off-diagonal Gell-Mann matrices *are* the $\sigma_x,\sigma_y$ of the
three two-level transitions. That is not an analogy — it is an identity.

In [2]:
PARES = [(0, 1), (0, 2), (1, 2)]
print(f"{'':10s}{'diagonal':>10s}{'level pairs it connects':>26s}")
for a in range(1, 9):
    M = np.asarray(GELLMANN[a], complex)
    pares = [p for p in PARES if abs(M[p]) > 1e-10]
    es_diag = np.max(np.abs(M - np.diag(np.diag(M)))) < 1e-10
    print(f"  lambda_{a}{str(es_diag):>10s}{str(pares) if pares else '-':>26s}")

# En cambio los generadores de momento angular manejan dos transiciones a la
# vez, que es justo lo que no es nativo: L_x = (lambda_1 + lambda_6)/sqrt(2).
M = Jx1.full()
print("\n  L_x connects", [p for p in PARES if abs(M[p]) > 1e-10],
      "-> not a native operation")

            diagonal   level pairs it connects
  lambda_1     False                  [(0, 1)]
  lambda_2     False                  [(0, 1)]
  lambda_3      True                         -
  lambda_4     False                  [(0, 2)]
  lambda_5     False                  [(0, 2)]
  lambda_6     False                  [(1, 2)]
  lambda_7     False                  [(1, 2)]
  lambda_8      True                         -

  L_x connects [(0, 1), (1, 2)] -> not a native operation


## The counting rule

A pool operator is a product of local factors, $G=\bigotimes_s M_s$ over its
support. Diagonalizing each factor turns the exponential into

$$e^{-i\theta G} = \Big(\bigotimes_s U_s\Big)\,e^{-i\theta\bigotimes_s D_s}\,\Big(\bigotimes_s U_s^\dagger\Big),$$

whose middle factor is diagonal and is realized by a ladder of $2(w-1)$ MS
gates around a single parametrized rotation. Writing $g(M)$ for the rotations
that diagonalize a local factor,

$$R = 2\sum_s g(M_s) + 1,\qquad \mathrm{MS} = 2(w-1)\qquad (w>1).$$

In [3]:
from funciones.utilidades_bp import conteo_compuertas

print("two operators actually selected by the algorithm, same support, same MS:\n")
for lbl, base, nom in (("((2, 2), (3, 8), (4, 6))", "gellmann",
                        "Gell-Mann   lambda_2 lambda_8 lambda_6"),
                       ("((1, 'y'), (3, 'x'), (4, 'z'))", "angular",
                        "angular     L_y L_x L_z")):
    c = conteo_compuertas(lbl, base=base)
    print(f"  {nom:36s} w={c['peso']}   R={c['r_dos_niveles']:3d}   MS={c['ms']}")

two operators actually selected by the algorithm, same support, same MS:

  Gell-Mann   lambda_2 lambda_8 lambda_6 w=3   R= 10   MS=4
  angular     L_y L_x L_z              w=3   R= 18   MS=4


## The tables of the paper

Table III compares the two pools at a moderate target precision; Table IV
compares Qudit-ADAPT against QAOA at a matched number of parameters.

In [4]:
import subprocess
print(subprocess.run([sys.executable, str(RAIZ / "cluster" / "tablas_apendice.py")],
                     capture_output=True, text=True, cwd=RAIZ).stdout)


TABLE II  --  pool comparison (n = 6 qutrits, instance G_2, l = 2)
  Pool basis           size  params     eps_rel  ent. gates  grad. meas
  Angular momentum     1600      16     3.6e-15          44       25600   (gradient_norm_below_epsilon)
  Gell-Mann            1292      30     4.3e-07          68       38760   (max_iteration_reached)

TABLE III  --  native gate decomposition at eps_rel <= 0.001
  Pool basis          params   R(i,j)    MS   total  grad. meas
  Angular momentum        16      218    44     262       25600
  Gell-Mann               21      192    46     238       27132

  Gell-Mann reduces the total native gate count from 262 to 238  (9 %)

Quoted in the text:
  Angular momentum   mean two-level rotations per generator: 13.62   needing Trotterization: 2/16
  Gell-Mann          mean two-level rotations per generator:  9.03   needing Trotterization: 0/30



In [5]:
salida = subprocess.run([sys.executable, str(RAIZ / "cluster" / "conteo_qaoa.py")],
                        capture_output=True, text=True, cwd=RAIZ).stdout
print(salida[salida.index("G_2  (ap"):])

G_2  (apéndice, 11 aristas)
                                     params    R(i,j)      MS    total
  Qudit-ADAPT  angular  l=2              16       218      44      262
  Qudit-ADAPT  Gell-Mann  (eps<1e-3)     21       192      46      238
  Qudit-ADAPT  Gell-Mann  (final)        30       271      68      339
  QAOA  (8 capas)                        16      1200     352     1552
  QAOA  (10 capas)                       21      1632     484     2116
  QAOA  (15 capas)                       30      2250     660     2910

  a igual presupuesto de parámetros:
    16 params: QAOA  1552  vs  angular    262   (5.9x)
    21 params: QAOA  2116  vs  Gell-Mann  238   (8.9x)

desglose por capa (G_2), en las dos decomposiciones del costo
  angular    mixer  R= 18  MS=  0   |  costo  R=264  MS= 44   |  capa  R=282  MS= 44
  gellmann   mixer  R= 18  MS=  0   |  costo  R=132  MS= 44   |  capa  R=150  MS= 44



QAOA needs between 5 and 9 times more native gates for the same number of
variational parameters. The reason is structural, not numerical: every QAOA
layer reapplies the mixer and the full cost Hamiltonian whether or not those
terms help, while the adaptive construction admits one generator at a time and
only when its gradient warrants it.

No QAOA optimization is involved in that comparison — the gate count of a QAOA
circuit is fixed by the graph and the number of layers alone.